In [1]:
import pickle

import numpy as np
import torch

import context
from src.quantum_env import QuantumEnv
from src.stategen import sample_haar_full, sample_haar_product
from src.util import srollout

In [2]:
PATH_4Q_AGENT = "../agents/4q-agent.pt"
PATH_5Q_AGENT = "../agents/5q-agent.pt"
PATH_6Q_AGENT = "../agents/6q-agent.pt"
PATH_8Q_AGENT = "../agents/8q-agent-epsi1e-2.pt"
PATH_10Q_AGENT = "../agents/5x5q-agent.pt"
PATH_12Q_AGENT = "../agents/4x4x4q-agent.pt"
PATH_15Q_AGENT = "../agents/5x5x5q-agent.pt"

In [3]:
data = []
np.random.seed(10)

Sample and solve 4q, 5q, 6q fully entangled Haar random states

In [4]:
num_qubits = (4,5,6)
agent_paths = (PATH_4Q_AGENT, PATH_5Q_AGENT, PATH_6Q_AGENT,)
max_steps = (10, 30, 80,)


for q, path, steps in zip(num_qubits, agent_paths, max_steps):
    agent = torch.load(path, map_location="cpu")
    n = 0
    print(f"{q} qubits:", end=' ')
    while n < 10:
        psi = sample_haar_full(q)
        actions, ents, _ = srollout(psi.copy(), agent, max_steps=steps, epsi=1e-3)
        if np.all(ents[-1] <= 1e-3):
            print(".", end='')
            data.append({
                "state": psi,
                "actions": actions,
                "entanglements": ents,
                "epsi": 1e-3
            })
            n += 1
        else:
            print(f"Failed to solve a {q}-qubit state", ents[-1])
    print()

4 qubits: ..........
5 qubits: ..........
6 qubits: ..........


Sample and solve 10q, 12q, 15q and 16q product states

In [5]:
num_qubits = (10, 12,)
subsystem_sizes = (5, 4, 5)
agent_paths = (PATH_10Q_AGENT, PATH_12Q_AGENT, PATH_15Q_AGENT)
max_steps = (60, 20, 90)


for q, s, path, steps in zip(num_qubits, subsystem_sizes, agent_paths, max_steps):
    agent = torch.load(path, map_location="cpu")
    n = 0
    print(f"{q} qubits:", end=' ')
    while n < 10:
        psi = sample_haar_product(q, min_subsystem_size=s, max_subsystem_size=s)
        actions, ents, _ = srollout(psi.copy(), agent, max_steps=steps, epsi=1e-2)
        if np.all(ents[-1] <= 1e-2):
            print(".", end='')
            data.append({
                "state": psi,
                "actions": actions,
                "entanglements": ents,
                "epsi": 1e-2
            })
            n += 1
        else:
            print(f"F", end='')
    print()

10 qubits: ..........
12 qubits: ..........


In [6]:
with open("../data/tests/rollouts.pickle", mode='wb') as f:
    pickle.dump(data, f)

In [7]:
data[10]["actions"]

array([[1, 4],
       [0, 1],
       [1, 2],
       [1, 4],
       [1, 3],
       [1, 2],
       [0, 4],
       [2, 3],
       [0, 3],
       [3, 4],
       [0, 4],
       [1, 3],
       [1, 4],
       [2, 3],
       [0, 3],
       [0, 1],
       [2, 4],
       [1, 4],
       [3, 4],
       [2, 3]])